# Deterministic Governance for OpenAI Function Calling

This cookbook demonstrates how to add **deterministic policy enforcement** to OpenAI function calling without adding another LLM to the governance path. You'll learn how to:

1. **Gate tool calls** with allowlist/blocklist policies before execution
2. **Enforce cost budgets** per session to prevent runaway spending
3. **Apply FREEZE rules** — immutable deny rules for destructive operations
4. **Track an audit trail** with correlation IDs for every governance decision
5. **Use governance modes** (ENFORCE, MONITOR, REPORT_ONLY) for safe rollout

**Key property:** All governance evaluation is deterministic — same input always produces the same decision. No LLM call in the safety path. Adds <5ms latency per evaluation.

## Why deterministic governance?

When OpenAI models use function calling, they decide which tools to invoke and with what arguments. In production, you need guarantees:

- The model can only call tools it's authorized to use
- Cost doesn't exceed budgets
- Destructive operations are blocked regardless of model behavior
- Every decision is auditable

Using another LLM to "check" tool calls introduces non-determinism — the same input might produce different safety decisions. A deterministic policy engine gives you reproducible, auditable guarantees.

## Prerequisites

```bash
pip install openai tealtiger
```

In [ ]:
# Install dependencies
!pip install openai tealtiger -q

In [ ]:
import os
import json
import uuid
import time
from openai import OpenAI

# Initialize OpenAI client
client = OpenAI(api_key=os.environ.get("OPENAI_API_KEY", "your-api-key-here"))

print("OpenAI client initialized")

## Part 1: The Problem — Ungoverned Function Calling

Let's start with a standard function calling setup. The model has access to several tools, including some that are potentially dangerous:

In [ ]:
# Define tools — mix of safe and dangerous operations
tools = [
    {
        "type": "function",
        "function": {
            "name": "search_web",
            "description": "Search the web for information",
            "parameters": {
                "type": "object",
                "properties": {"query": {"type": "string"}},
                "required": ["query"]
            }
        }
    },
    {
        "type": "function",
        "function": {
            "name": "read_file",
            "description": "Read a file from the filesystem",
            "parameters": {
                "type": "object",
                "properties": {"path": {"type": "string"}},
                "required": ["path"]
            }
        }
    },
    {
        "type": "function",
        "function": {
            "name": "delete_file",
            "description": "Delete a file from the filesystem",
            "parameters": {
                "type": "object",
                "properties": {"path": {"type": "string"}},
                "required": ["path"]
            }
        }
    },
    {
        "type": "function",
        "function": {
            "name": "execute_sql",
            "description": "Execute a SQL query against the database",
            "parameters": {
                "type": "object",
                "properties": {"query": {"type": "string"}, "database": {"type": "string"}},
                "required": ["query", "database"]
            }
        }
    },
    {
        "type": "function",
        "function": {
            "name": "send_email",
            "description": "Send an email to a recipient",
            "parameters": {
                "type": "object",
                "properties": {
                    "to": {"type": "string"},
                    "subject": {"type": "string"},
                    "body": {"type": "string"}
                },
                "required": ["to", "subject", "body"]
            }
        }
    }
]

print(f"Defined {len(tools)} tools: {[t['function']['name'] for t in tools]}")

Without governance, a jailbroken or misaligned model could call `delete_file` or `execute_sql` with destructive queries. Let's add a governance layer.

## Part 2: Adding Deterministic Governance with TealTiger

TealTiger provides a deterministic policy engine that evaluates every tool call before execution. Here's how to set it up:

In [ ]:
from dataclasses import dataclass, field
from typing import Any, Dict, List, Optional, Set
from enum import Enum


class GovernanceAction(str, Enum):
    ALLOW = "ALLOW"
    DENY = "DENY"


class GovernanceMode(str, Enum):
    ENFORCE = "ENFORCE"      # Block denied actions
    MONITOR = "MONITOR"      # Allow all, log violations
    REPORT_ONLY = "REPORT_ONLY"  # Allow all, generate reports


@dataclass
class GovernanceDecision:
    action: GovernanceAction
    tool_name: str
    tool_args: Dict[str, Any]
    reason: str
    correlation_id: str
    evaluation_time_ms: float
    risk_score: int = 0
    triggered_policies: List[str] = field(default_factory=list)


class TealTigerGovernance:
    """Deterministic governance engine for OpenAI function calling.
    
    No LLM in the governance path. All evaluation is rule-based.
    Typical evaluation time: <1ms.
    """

    def __init__(
        self,
        tool_allowlist: Optional[List[str]] = None,
        freeze_tools: Optional[List[str]] = None,
        max_session_cost: Optional[float] = None,
        max_calls: Optional[int] = None,
        mode: GovernanceMode = GovernanceMode.ENFORCE,
    ):
        self._allowlist = set(tool_allowlist) if tool_allowlist else None
        self._freeze_tools = set(freeze_tools or [])
        self._max_session_cost = max_session_cost
        self._max_calls = max_calls
        self._mode = mode
        
        # Session state
        self._session_cost = 0.0
        self._call_count = 0
        self._evidence: List[GovernanceDecision] = []

    def evaluate(self, tool_name: str, tool_args: Dict[str, Any]) -> GovernanceDecision:
        """Evaluate a tool call against all configured policies.
        
        Returns a GovernanceDecision in <1ms. Deterministic: same input → same output.
        """
        start = time.perf_counter()
        correlation_id = str(uuid.uuid4())
        triggered = []

        # 1. FREEZE check (always enforced, regardless of mode)
        if tool_name in self._freeze_tools:
            decision = GovernanceDecision(
                action=GovernanceAction.DENY,
                tool_name=tool_name,
                tool_args=tool_args,
                reason=f"FREEZE: '{tool_name}' is permanently blocked",
                correlation_id=correlation_id,
                evaluation_time_ms=(time.perf_counter() - start) * 1000,
                risk_score=100,
                triggered_policies=["freeze"],
            )
            self._evidence.append(decision)
            return decision

        # 2. Tool allowlist check
        if self._allowlist and tool_name not in self._allowlist:
            triggered.append("tool_allowlist")
            if self._mode == GovernanceMode.ENFORCE:
                decision = GovernanceDecision(
                    action=GovernanceAction.DENY,
                    tool_name=tool_name,
                    tool_args=tool_args,
                    reason=f"Tool '{tool_name}' not in allowlist",
                    correlation_id=correlation_id,
                    evaluation_time_ms=(time.perf_counter() - start) * 1000,
                    risk_score=80,
                    triggered_policies=triggered,
                )
                self._evidence.append(decision)
                return decision

        # 3. Rate limit check
        if self._max_calls and self._call_count >= self._max_calls:
            triggered.append("rate_limit")
            if self._mode == GovernanceMode.ENFORCE:
                decision = GovernanceDecision(
                    action=GovernanceAction.DENY,
                    tool_name=tool_name,
                    tool_args=tool_args,
                    reason=f"Rate limit: {self._call_count}/{self._max_calls} calls used",
                    correlation_id=correlation_id,
                    evaluation_time_ms=(time.perf_counter() - start) * 1000,
                    risk_score=70,
                    triggered_policies=triggered,
                )
                self._evidence.append(decision)
                return decision

        # 4. Cost limit check
        if self._max_session_cost and self._session_cost >= self._max_session_cost:
            triggered.append("cost_limit")
            if self._mode == GovernanceMode.ENFORCE:
                decision = GovernanceDecision(
                    action=GovernanceAction.DENY,
                    tool_name=tool_name,
                    tool_args=tool_args,
                    reason=f"Cost limit: ${self._session_cost:.2f}/${self._max_session_cost:.2f}",
                    correlation_id=correlation_id,
                    evaluation_time_ms=(time.perf_counter() - start) * 1000,
                    risk_score=60,
                    triggered_policies=triggered,
                )
                self._evidence.append(decision)
                return decision

        # All checks passed
        self._call_count += 1
        decision = GovernanceDecision(
            action=GovernanceAction.ALLOW,
            tool_name=tool_name,
            tool_args=tool_args,
            reason="Compliant with all policies",
            correlation_id=correlation_id,
            evaluation_time_ms=(time.perf_counter() - start) * 1000,
            risk_score=0,
            triggered_policies=triggered,
        )
        self._evidence.append(decision)
        return decision

    def record_cost(self, cost: float):
        """Record cost after a successful tool call."""
        self._session_cost += cost

    @property
    def evidence(self) -> List[GovernanceDecision]:
        return list(self._evidence)

    @property
    def summary(self) -> Dict[str, Any]:
        return {
            "total_evaluations": len(self._evidence),
            "allowed": sum(1 for d in self._evidence if d.action == GovernanceAction.ALLOW),
            "denied": sum(1 for d in self._evidence if d.action == GovernanceAction.DENY),
            "session_cost": self._session_cost,
            "mode": self._mode.value,
        }


print("✅ TealTigerGovernance engine defined")

## Part 3: Governed Function Calling Loop

Now let's build a function calling loop with governance at the tool boundary. The governance check happens **after** the model decides to call a tool but **before** the tool actually executes:

In [ ]:
# Initialize governance with policies
governance = TealTigerGovernance(
    tool_allowlist=["search_web", "read_file", "send_email"],  # No delete_file, no execute_sql
    freeze_tools=["delete_file"],  # FREEZE: always blocked, even in MONITOR mode
    max_session_cost=5.00,  # $5 budget per session
    max_calls=20,  # Max 20 tool calls per session
    mode=GovernanceMode.ENFORCE,
)


def execute_tool(tool_name: str, tool_args: Dict[str, Any]) -> str:
    """Simulate tool execution. In production, these would be real implementations."""
    if tool_name == "search_web":
        return f"Search results for: {tool_args.get('query', '')}"
    elif tool_name == "read_file":
        return f"Contents of {tool_args.get('path', '')}: [file data]"
    elif tool_name == "send_email":
        return f"Email sent to {tool_args.get('to', '')}"
    elif tool_name == "delete_file":
        return f"DELETED: {tool_args.get('path', '')}"
    elif tool_name == "execute_sql":
        return f"SQL result: {tool_args.get('query', '')}"
    return "Unknown tool"


def governed_function_calling(user_message: str, governance: TealTigerGovernance):
    """Run a function calling loop with deterministic governance at the tool boundary."""
    messages = [{"role": "user", "content": user_message}]
    
    print(f"\n{'='*60}")
    print(f"User: {user_message}")
    print(f"{'='*60}")

    # Call the model
    response = client.chat.completions.create(
        model="gpt-4o-mini",
        messages=messages,
        tools=tools,
    )

    message = response.choices[0].message

    # If no tool calls, return the response directly
    if not message.tool_calls:
        print(f"\nAssistant: {message.content}")
        return message.content

    # Process each tool call through governance
    messages.append(message)
    
    for tool_call in message.tool_calls:
        tool_name = tool_call.function.name
        tool_args = json.loads(tool_call.function.arguments)

        print(f"\n🔧 Model wants to call: {tool_name}({tool_args})")

        # ┌─────────────────────────────────────────────┐
        # │  GOVERNANCE GATE — deterministic evaluation  │
        # └─────────────────────────────────────────────┘
        decision = governance.evaluate(tool_name, tool_args)

        print(f"   ⚖️  Decision: {decision.action.value} ({decision.evaluation_time_ms:.3f}ms)")
        print(f"   📋 Reason: {decision.reason}")
        print(f"   🔗 Correlation ID: {decision.correlation_id[:8]}...")

        if decision.action == GovernanceAction.DENY:
            # Tool call blocked — return denial to the model
            tool_result = f"[GOVERNANCE DENIED] {decision.reason}"
            print(f"   🚫 BLOCKED")
        else:
            # Tool call allowed — execute normally
            tool_result = execute_tool(tool_name, tool_args)
            print(f"   ✅ EXECUTED: {tool_result[:50]}...")

        messages.append({
            "role": "tool",
            "tool_call_id": tool_call.id,
            "content": tool_result,
        })

    # Get final response
    final_response = client.chat.completions.create(
        model="gpt-4o-mini",
        messages=messages,
        tools=tools,
    )

    final_message = final_response.choices[0].message.content
    print(f"\n💬 Assistant: {final_message}")
    return final_message


print("✅ Governed function calling loop defined")

## Part 4: Testing Governance in Action

### Test 1: Allowed tool call

In [ ]:
# This should be ALLOWED — search_web is in the allowlist
governed_function_calling(
    "Search the web for the latest news about AI governance",
    governance
)

### Test 2: Blocked tool call (not in allowlist)

In [ ]:
# This should be DENIED — execute_sql is NOT in the allowlist
governed_function_calling(
    "Run this SQL query: DROP TABLE users;",
    governance
)

### Test 3: FREEZE rule (always blocked)

In [ ]:
# This should be DENIED by FREEZE — delete_file is frozen regardless of mode
governed_function_calling(
    "Delete the file at /etc/passwd",
    governance
)

## Part 5: Governance Evidence Trail

Every governance decision is recorded with a correlation ID, timestamp, and evaluation time. This creates a full audit trail:

In [ ]:
# Review the governance evidence trail
print("\n" + "="*60)
print("GOVERNANCE SESSION SUMMARY")
print("="*60)
print(json.dumps(governance.summary, indent=2))

print("\n" + "-"*60)
print("DECISION LOG")
print("-"*60)

for i, decision in enumerate(governance.evidence, 1):
    emoji = "✅" if decision.action == GovernanceAction.ALLOW else "🚫"
    print(f"\n{i}. {emoji} {decision.action.value} | {decision.tool_name}")
    print(f"   Reason: {decision.reason}")
    print(f"   Risk Score: {decision.risk_score}/100")
    print(f"   Eval Time: {decision.evaluation_time_ms:.3f}ms")
    print(f"   Correlation: {decision.correlation_id}")
    if decision.triggered_policies:
        print(f"   Triggered: {decision.triggered_policies}")

## Part 6: Governance Modes for Safe Rollout

TealTiger supports three governance modes for progressive deployment:

| Mode | Behavior | Use Case |
|------|----------|----------|
| `ENFORCE` | Block denied actions | Production |
| `MONITOR` | Allow all, log violations | Staging / testing |
| `REPORT_ONLY` | Allow all, generate reports | Initial rollout |

Start in MONITOR mode to observe what *would* be blocked, then switch to ENFORCE when confident:

In [ ]:
# MONITOR mode — observe violations without blocking
monitor_governance = TealTigerGovernance(
    tool_allowlist=["search_web"],
    freeze_tools=["delete_file"],
    mode=GovernanceMode.MONITOR,  # Won't block, just logs
)

# This would be DENIED in ENFORCE mode, but is ALLOWED in MONITOR
decision = monitor_governance.evaluate("execute_sql", {"query": "SELECT * FROM users"})
print(f"Mode: MONITOR")
print(f"Tool: execute_sql (not in allowlist)")
print(f"Decision: {decision.action.value}")
print(f"Triggered policies: {decision.triggered_policies}")
print(f"\nNote: In MONITOR mode, the tool call proceeds but the violation is logged.")
print(f"FREEZE rules are ALWAYS enforced regardless of mode:")

# FREEZE still blocks even in MONITOR mode
freeze_decision = monitor_governance.evaluate("delete_file", {"path": "/data"})
print(f"\nFREEZE check on delete_file: {freeze_decision.action.value} ← Always enforced")

## Part 7: Performance — Sub-Millisecond Governance

A key advantage of deterministic governance: evaluation is fast enough to be invisible in the request path.

In [ ]:
# Benchmark governance evaluation latency
perf_governance = TealTigerGovernance(
    tool_allowlist=["search_web", "read_file", "send_email"],
    freeze_tools=["delete_file", "execute_sql"],
    max_session_cost=100.0,
    max_calls=1000,
    mode=GovernanceMode.ENFORCE,
)

# Run 1000 evaluations and measure
latencies = []
for i in range(1000):
    start = time.perf_counter()
    perf_governance.evaluate("search_web", {"query": f"test query {i}"})
    latencies.append((time.perf_counter() - start) * 1000)

latencies.sort()
print(f"Governance Evaluation Latency (1000 calls):")
print(f"  P50: {latencies[499]:.4f}ms")
print(f"  P95: {latencies[949]:.4f}ms")
print(f"  P99: {latencies[989]:.4f}ms")
print(f"  Max: {latencies[-1]:.4f}ms")
print(f"\nFor comparison: a typical OpenAI API call takes 200-2000ms.")
print(f"Governance adds <{latencies[989]:.2f}ms — effectively invisible.")

## Part 8: Using the Published `tealtiger` Package

The examples above show the governance logic inline for clarity. In production, use the published `tealtiger` package which includes the full v1.3 engine with additional features:

- **NHI (Non-Human Identity)** — Per-agent identity and scope enforcement
- **TealProof** — Cryptographic governance receipts (Merkle + RFC 3161)
- **SARIF Export** — Structured evidence export for compliance
- **12 LLM Providers** — Works with OpenAI, Anthropic, Gemini, Bedrock, Azure, and more

```python
from tealtiger.core.engine.v1_3 import TealEngineV13, TealEngineV13Options
from tealtiger.core.engine.v1_3.types import (
    GovernanceRequest, GovernanceContext, FreezeRule, NHIDescriptor
)

# Full v1.3 engine with FREEZE, NHI, attestation, ZSP
engine = TealEngineV13(TealEngineV13Options(
    freeze_rules=[
        FreezeRule(id="fr-1", action_pattern="delete_*", reason="Destructive ops blocked"),
        FreezeRule(id="fr-2", tool_pattern="execute_sql", reason="Direct SQL blocked"),
    ],
    nhi_inventory=...,  # Agent identity registry
))

# Evaluate with full governance pipeline
decision = await engine.evaluate(
    GovernanceRequest(
        correlation_id=str(uuid.uuid4()),
        action_class="TOOL_INVOKE",
        tool="search_web",
        nhi_identity=NHIDescriptor(agent_id="agent-001", ...),
    ),
    GovernanceContext(environment="production"),
)
```

## Summary

| Property | Value |
|----------|-------|
| Governance latency | <1ms P99 |
| LLM in safety path | None (deterministic) |
| Same input → same output | Always |
| Audit trail | Full, with correlation IDs |
| FREEZE rules | Immutable, always enforced |
| Deployment modes | ENFORCE → MONITOR → REPORT_ONLY |

**Key insight:** Model alignment and governance are complementary layers. The model may be jailbroken, but if the governance layer says DENY based on policy rules, the destructive tool call never executes.

## Resources

- [TealTiger GitHub](https://github.com/agentguard-ai/tealtiger) — Apache 2.0
- [TealTiger Documentation](https://docs.tealtiger.ai)
- [PyPI: tealtiger](https://pypi.org/project/tealtiger/)
- [LangChain Middleware Integration](https://pypi.org/project/langchain-tealtiger/)